In [1]:
# ── Cell 0 — GPU check ────────────────────────────────────────────────
!nvidia-smi -L
import torch; assert torch.cuda.is_available(), "Set Runtime ▸ Change runtime type ▸ GPU"
print("CUDA:", torch.version.cuda, "| device:", torch.cuda.get_device_name(0))

GPU 0: NVIDIA L4 (UUID: GPU-21813cc1-69ca-a14b-296e-79cb1aaaa979)
CUDA: 12.8 | device: NVIDIA L4


In [2]:
# ── Cell 1 — mount Drive + cache dir ──────────────────────────────────
from google.colab import drive; drive.mount('/content/drive')
import os
DRIVE = "/content/drive/MyDrive/haidc_m2"          # persists across sessions
os.makedirs(f"{DRIVE}/kaggle", exist_ok=True)
os.makedirs(f"{DRIVE}/artifacts", exist_ok=True)
print("cache:", DRIVE)

Mounted at /content/drive
cache: /content/drive/MyDrive/haidc_m2


In [3]:
# ── Cell 2 — clone the branch into /content/repo, then cd into it ──────
# GITHUB_TOKEN is read from Colab Secrets (🔑 left sidebar) — no re-pasting.
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
USERNAME, REPO_NAME = "Alessandro-vecchi", "Rethinking-human-AI-decision-making"
BRANCH_NAME = "claude/charming-gates-hzjyhx"
!rm -rf /content/repo
!git clone -b {BRANCH_NAME} --single-branch -q https://{GITHUB_TOKEN}@github.com/{USERNAME}/{REPO_NAME}.git /content/repo
%cd /content/repo
!git log --oneline -1

/content/repo
4bfe44c (HEAD -> claude/charming-gates-hzjyhx, origin/claude/charming-gates-hzjyhx) M4: L2D-Okati arm — Option B (frozen classifier) oracle frontier + learned rejector (staged)


In [4]:
# ── Cell 3 — deps (use Colab's CUDA torch; install the rest) ──────────
# NOTE: this deviates from the pinned CPU torch 2.2.2 — expected per DECISIONS
# 2026-06-26 env note. Record the GPU torch version + a fresh lockfile hash later.
!pip -q install pandas pyarrow pyyaml scipy scikit-learn tqdm pillow
!pip -q install -e . --no-deps
import torch; print("torch", torch.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for haidc (pyproject.toml) ... done
torch 2.11.0+cu128


In [6]:
# ── Cell 4 — Kaggle creds + cached download to Drive (one upload, ever) ─
# One-time: accept the rules at kaggle.com/c/galaxy-zoo-the-galaxy-challenge/rules.
# Creds resolve from the Drive cache first; upload kaggle.json only if absent.
import os, json, pathlib, shutil
CRED = pathlib.Path(f"{DRIVE}/kaggle/kaggle.json")
if not CRED.exists():
    from google.colab import files; files.upload(); shutil.copy("kaggle.json", CRED)
creds = json.load(open(CRED))
os.environ["KAGGLE_USERNAME"] = creds["username"].strip()   # env wins over any stale file
os.environ["KAGGLE_KEY"]      = creds["key"].strip()
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(CRED, "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
!pip -q install --upgrade kaggle
C = "galaxy-zoo-the-galaxy-challenge"
for f in ["images_training_rev1.zip", "training_solutions_rev1.zip"]:
    if pathlib.Path(f"{DRIVE}/kaggle/{f}").exists():
        print("cached:", f)
    else:
        get_ipython().system(f"kaggle competitions download -c {C} -f {f} -p {DRIVE}/kaggle")
!ls -lh {DRIVE}/kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 25.9 MB/s eta 0:00:00
cached: images_training_rev1.zip
cached: training_solutions_rev1.zip
total 797M
-rw------- 1 root root 792M Dec 11  2019 images_training_rev1.zip
-rw------- 1 root root   74 Jun 27 01:31 kaggle.json
-rw------- 1 root root 4.7M Dec 11  2019 training_solutions_rev1.zip


In [7]:
# ── Cell 5 — fast local extract (Drive→local SSD, then unzip) ─────────
import glob
!cp {DRIVE}/kaggle/images_training_rev1.zip /content/imgs.zip
!unzip -q -o /content/imgs.zip -d /content/repo/data/raw/     # -> data/raw/images_training_rev1/
!cp {DRIVE}/kaggle/training_solutions_rev1.zip /content/sol.zip
!unzip -q -o /content/sol.zip -d /content/repo/data/raw/
n = len(glob.glob("/content/repo/data/raw/images_training_rev1/*.jpg"))
print("images:", n); assert n > 60000, "extract looks wrong"

images: 61578


In [8]:
# ── Cell 6 — label table (upload once to Drive, then reuse) ───────────
# label_table.parquet is git-ignored; bring it from your local data/ folder.
import pathlib
DST = "/content/repo/data/label_table.parquet"
if pathlib.Path(f"{DRIVE}/artifacts/label_table.parquet").exists():
    !cp {DRIVE}/artifacts/label_table.parquet {DST}
else:
    from google.colab import files; files.upload()             # pick label_table.parquet
    !cp label_table.parquet {DST} && cp {DST} {DRIVE}/artifacts/label_table.parquet
import pandas as pd
print(pd.read_parquet(DST).shape, "| split_manifest committed:",
      pathlib.Path("/content/repo/data/split_manifest.json").exists())

(4621, 8) | split_manifest committed: True


In [ ]:
# ── Cell 7 — fidelity check: print Okati's exact image transform ──────
!git clone -q https://github.com/Networks-Learning/differentiable-learning-under-triage /content/okati || true
!grep -nE "Resize|Crop|Normalize|transforms|resize|reshape|224" /content/okati/Galaxy-zoo/prepare_data.py | head -40
# Compare against backbone.py: Resize(256)+CenterCrop(224)+ImageNet norm. If Okati differs,
# that's the first thing to fix if accuracy misses ~0.83.

7:from skimage.transform import rescale, resize, downscale_local_mean
19:X = np.zeros((num_samples,3,224,224),dtype='float')
38:        from torchvision import transforms
41:        preprocess = transforms.Compose([
42:            transforms.Resize(256),
43:            transforms.CenterCrop(224),
44:            transforms.ToTensor(),
45:            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),


In [ ]:
# ── Cell 8 — run M2 (CUBLAS env makes deterministic algos legal on CUDA) ─
import os, time
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
t = time.time()
!cd /content/repo && python -m haidc.arms.backbone --config configs/backbone.yaml
print(f"elapsed {(time.time()-t)/60:.1f} min")

backbone: trained on 3234 imgs, val 693, scored 694 test imgs
AI-alone test accuracy @ thr 0.5 = 0.8285 (95% CI [0.8011, 0.8573])
best-VAL epoch = 58/120 | val acc = 0.8268
train accuracy = 0.9604 | final train loss = 0.0000 | train-test gap = +0.1319
train-acc curve: first=0.553 max=1.000 last=1.000  (verdict: climb to ~0.93+ => under-trained; plateau ~0.80-0.85 => label ceiling)
scores -> results/backbone_scores.parquet   artifact -> results/backbone.pt
elapsed 46.2 min


In [ ]:
# ── Cell 9 — read verdict diagnostics + persist (scores, manifest, AND weights) ───
import json, shutil
m = json.load(open("/content/repo/results/backbone_run.json"))
test_acc, train_acc = m["ai_alone_accuracy"], m["train_accuracy"]
tac = m["train_acc_curve"]
print(f"AI-alone TEST acc = {test_acc:.4f}  CI {m['ai_alone_accuracy_ci']}")
print(f"best-VAL epoch = {m['best_epoch']}/{len(tac)} | VAL acc = {m['val_accuracy']:.4f}")
print(f"TRAIN acc = {train_acc:.4f} | final train loss = {m['final_train_loss']:.4f} "
      f"| train-test gap = {train_acc - test_acc:+.4f}")
print(f"train-acc curve: first={tac[0]:.3f}  max={max(tac):.3f}  last={tac[-1]:.3f}")
# VERDICT (DECISIONS 2026-06-27): train acc climbs to ~0.93+ AND test improves => was UNDER-TRAINED
# (freeze this run). train acc PLATEAUS ~0.80-0.85 well before epoch 120 => LABEL/CONSENSUS CEILING,
# not an optimization bug (accept; freeze the best-VAL checkpoint). 0.83 is NOT required (HANDOFF §1).
print("anchor 0.83:", "PASS" if 0.80 <= test_acc <= 0.86 else "below -> read the curve, not the anchor")
# Persist ALL THREE artifacts: scores.parquet (M3+ interface) + run.json + backbone.pt (the weights —
# git-ignored and Colab-ephemeral; download so they SURVIVE this session — the whole point of re-running).
for f in ["backbone_scores.parquet", "backbone_run.json", "backbone.pt"]:
    shutil.copy(f"/content/repo/results/{f}", f"{DRIVE}/artifacts/{f}")
from google.colab import files
for f in ["backbone_scores.parquet", "backbone_run.json", "backbone.pt"]:
    files.download(f"/content/repo/results/{f}")

AI-alone TEST acc = 0.8285  CI [0.8011167146974063, 0.8573487031700289]
best-VAL epoch = 58/120 | VAL acc = 0.8268
TRAIN acc = 0.9604 | final train loss = 0.0000 | train-test gap = +0.1319
train-acc curve: first=0.553  max=1.000  last=1.000
anchor 0.83: PASS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# ── Cell 10 — Stage-B infra: export 2048-d embeddings from the FROZEN backbone ────
# Shared by the L2D-Okati(learned) arm (M4) and L2D-Mozannar (M5). Reads results/backbone.pt
# (NO retrain — preserves the one-shared-backbone invariant) + the extracted images (Cell 5) and
# writes results/backbone_embeddings.parquet = [GalaxyID, split, score, e0..e2047] for train/val/test.
# Prereqs in a FRESH session: run Cells 0-6 (repo + images + label table), then this cell.
import os, shutil, time, pathlib
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# bring the frozen weights back from Drive if this session didn't just train them
PT = "/content/repo/results/backbone.pt"
if not pathlib.Path(PT).exists():
    os.makedirs("/content/repo/results", exist_ok=True)
    shutil.copy(f"{DRIVE}/artifacts/backbone.pt", PT)
t = time.time()
!cd /content/repo && python -m haidc.arms.backbone --config configs/backbone.yaml --export-embeddings
print(f"elapsed {(time.time()-t)/60:.1f} min")
# persist + download (git-ignored, Colab-ephemeral)
EMB = "/content/repo/results/backbone_embeddings.parquet"
shutil.copy(EMB, f"{DRIVE}/artifacts/backbone_embeddings.parquet")
import pandas as pd
df = pd.read_parquet(EMB)
print(df.shape, "| splits:", df.split.value_counts().to_dict(), "| feat cols:", sum(c[0]=='e' and c[1:].isdigit() for c in df.columns))
from google.colab import files; files.download(EMB)

embeddings: 4621 rows x 2048 feats (train 3234, val 693, test 694) on cuda -> results/backbone_embeddings.parquet
elapsed 0.6 min
(4621, 2051) | splits: {'train': 3234, 'test': 694, 'val': 693} | feat cols: 2048


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>